In [1]:
#%pip install pandas==2.2.3
#%pip install git+https://github.com/pydata/pandas-datareader.git
#%pip install yfinance numpy matplotlib
#%pip install fredapi
#%pip install pyarrow


In [2]:
#Imports necesarios para mdescargar los datos y manejar el dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from pandas_datareader import data as pdr 
from fredapi import Fred


In [3]:
def imputar_valores_faltantes(dataframe, nombre_columna=None):

    if isinstance(dataframe, pd.Series):
        serie_imputada = dataframe.copy()
    elif isinstance(dataframe, pd.DataFrame):
        if nombre_columna is not None:
            serie_imputada = dataframe[nombre_columna].copy()
        elif dataframe.shape[1] == 1:
            serie_imputada = dataframe.iloc[:, 0].copy()
        else:
            raise ValueError("Debes especificar 'nombre_columna'")
    else:
        raise TypeError("Input debe ser DataFrame o Series")

    serie_imputada = pd.to_numeric(serie_imputada, errors='coerce')

    valores_cero = (serie_imputada == 0)
    valores_nan = serie_imputada.isna()
    total_a_imputar = valores_cero.sum() + valores_nan.sum()

    serie_imputada[valores_cero] = np.nan

    if total_a_imputar == 0:
        return serie_imputada

    metodo1 = serie_imputada.interpolate(method='linear', limit_direction='both')

    media_3m = serie_imputada.rolling(window=3, center=True, min_periods=1).mean()
    media_6m = serie_imputada.rolling(window=6, center=True, min_periods=1).mean()
    metodo2 = media_3m * 0.7 + media_6m * 0.3

    ffill = serie_imputada.ffill()
    bfill = serie_imputada.bfill()
    metodo3 = ffill.fillna(bfill)

    mask = serie_imputada.isna()
    valores_nulos = np.where(mask)[0]

    for valor in valores_nulos:
        valores = []
        pesos = []

        if valor < len(metodo1) and not pd.isna(metodo1.iloc[valor]):
            valores.append(metodo1.iloc[valor])
            pesos.append(0.50)

        if valor < len(metodo2) and not pd.isna(metodo2.iloc[valor]):
            valores.append(metodo2.iloc[valor])
            pesos.append(0.30)

        if valor < len(metodo3) and not pd.isna(metodo3.iloc[valor]):
            valores.append(metodo3.iloc[valor])
            pesos.append(0.20)

        if valores:
            pesos_normalizados = np.array(pesos) / np.sum(pesos)
            valor_imputado = np.average(valores, weights=pesos_normalizados)
            serie_imputada.iloc[valor] = valor_imputado

    if serie_imputada.isna().any():
        serie_imputada = serie_imputada.interpolate(method='linear', limit_direction='both')

    if serie_imputada.isna().any():
        serie_imputada = serie_imputada.ffill()
        serie_imputada = serie_imputada.bfill()

    return serie_imputada

In [4]:
dataset = pd.DataFrame() #dataset global

inicio = '2001-12-01' #fecha en la que empezamos a descargar datos
fin = '2026-03-01' #fecga en la que dejamos de descargar datos

sp500 = yf.download('^GSPC', start=inicio, end=fin, interval='1mo')['Close'] #Descargamos los datos de cierredel sp500
sp500 = sp500.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes
sp500["sp500_return"] = sp500.pct_change() #Calculamos el retorno mensual del sp500 y lo guardamos en una nueva columna
sp500 = sp500.dropna() #Eliminamos los valores nulos
sp500 = sp500[["sp500_return"]]
#sp500.info()

sp500 = sp500.reset_index()
dataset = sp500 #Asignamos el dataset global al dataset del sp500, que es el que vamos a ir ampliando con las demás variables
dataset.head()

[*********************100%***********************]  1 of 1 completed

1 Failed download:
['^GSPC']: TypeError("'NoneType' object is not subscriptable")


Ticker,Date,sp500_return


In [5]:
eurodollar = yf.download('EURUSD=X', start=inicio, end=fin, interval='1mo')['Close'] #Descargamos los datos de cierre del eurodolar
#eurodollar.info()
eurodollar = eurodollar.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes
eurodollar = eurodollar.rename(columns={"Close": "EUR/USD"})
eurodollar = eurodollar.reset_index()

dataset = pd.merge(dataset,eurodollar, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del eurodolar, uniendo por la columna "Date" y quedándonos solo con las filas que tengan fecha en ambos datasets

dataset.head()

[*********************100%***********************]  1 of 1 completed

1 Failed download:
['EURUSD=X']: TypeError("'NoneType' object is not subscriptable")


Ticker,Date,sp500_return,EURUSD=X


In [6]:
vix = yf.download('^VIX', start=inicio, end=fin, interval='1mo')['Close'] #Descargamos los datos de cierre del vix
vix.info()
vix = vix.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el vix ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
vix = vix.rename(columns={"Close":"vix"})
vix = vix.reset_index()
#vix.head()

dataset = pd.merge(dataset, vix, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del vix, uniendo por la columna "Date" y quedándonos solo con las filas que tengan fecha en ambos datasets
dataset.head()

[*********************100%***********************]  1 of 1 completed

1 Failed download:
['^VIX']: TypeError("'NoneType' object is not subscriptable")


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 0 entries
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ^VIX    0 non-null      float64
dtypes: float64(1)
memory usage: 0.0 bytes


Ticker,Date,sp500_return,EURUSD=X,^VIX


In [7]:
petroleo = yf.download('CL=F', start=inicio, end=fin, interval='1mo') 
petroleo.info()
petroleo.columns = petroleo.columns.get_level_values(0) #El dataset del petroleo tiene un multiindex en las columnas, con el nombre del ticker y el nombre de la variable, por lo que tenemos que eliminar el nivel del ticker para quedarnos solo con el nombre de la variable
petroleo = petroleo.resample("ME").last()
petroleo = petroleo[['Close']].copy()
petroleo['Close'] = imputar_valores_faltantes(petroleo, 'Close')
petroleo["petroleo_return"] = petroleo["Close"].pct_change()
petroleo = petroleo[["petroleo_return"]]
petroleo = petroleo.reset_index()

dataset = pd.merge(dataset, petroleo, on="Date", how="inner")

[*********************100%***********************]  1 of 1 completed

1 Failed download:
['CL=F']: TypeError("'NoneType' object is not subscriptable")


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 0 entries
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   (Adj Close, CL=F)  0 non-null      float64
 1   (Close, CL=F)      0 non-null      float64
 2   (High, CL=F)       0 non-null      float64
 3   (Low, CL=F)        0 non-null      float64
 4   (Open, CL=F)       0 non-null      float64
 5   (Volume, CL=F)     0 non-null      float64
dtypes: float64(6)
memory usage: 0.0 bytes


In [8]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             0 non-null      datetime64[ns]
 1   sp500_return     0 non-null      float64       
 2   EURUSD=X         0 non-null      float64       
 3   ^VIX             0 non-null      float64       
 4   petroleo_return  0 non-null      float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 132.0 bytes


In [9]:
oro = yf.download('GC=F', start=inicio, end=fin, interval='1mo')
oro.info()  
oro.columns = oro.columns.get_level_values(0)
oro = oro.resample("ME").last()
oro = oro[['Close']].copy()
oro['Close'] = imputar_valores_faltantes(oro, 'Close')
oro["oro_return"] = oro["Close"].pct_change()
oro = oro[["oro_return"]]
oro = oro.reset_index()

dataset = pd.merge(dataset, oro, on="Date", how="inner")

[*********************100%***********************]  1 of 1 completed

1 Failed download:
['GC=F']: TypeError("'NoneType' object is not subscriptable")


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 0 entries
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   (Adj Close, GC=F)  0 non-null      float64
 1   (Close, GC=F)      0 non-null      float64
 2   (High, GC=F)       0 non-null      float64
 3   (Low, GC=F)        0 non-null      float64
 4   (Open, GC=F)       0 non-null      float64
 5   (Volume, GC=F)     0 non-null      float64
dtypes: float64(6)
memory usage: 0.0 bytes


In [10]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             0 non-null      datetime64[ns]
 1   sp500_return     0 non-null      float64       
 2   EURUSD=X         0 non-null      float64       
 3   ^VIX             0 non-null      float64       
 4   petroleo_return  0 non-null      float64       
 5   oro_return       0 non-null      float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 132.0 bytes


In [ ]:
fred = Fred(api_key="af59516007957e0c89174b4d426736ca")

tipos_fed = fred.get_series("FEDFUNDS", observation_start=inicio, observation_end=fin)
tipos_fed = tipos_fed.to_frame()

tipos_fed.info()
tipos_fed = tipos_fed.resample("ME").last()
tipos_fed = tipos_fed.rename(columns={tipos_fed.columns[0]: "tipos_fed"})
tipos_fed = tipos_fed.reset_index()
tipos_fed = tipos_fed.rename(columns={"index":"Date"})

dataset = pd.merge(dataset, tipos_fed, on="Date", how="inner")

In [ ]:
dataset.head()

In [ ]:
ipc_usa = fred.get_series("CPIAUCSL", observation_start=inicio, observation_end=fin) #Descargamos los datos del IPC de Estados Unidos desde la base de datos de FRED, utilizando el código CPIAUCSL que corresponde al IPC de Estados Unidos, y especificando el rango de fechas que queremos descargar
ipc_usa = ipc_usa.to_frame() 

ipc_usa.info()
ipc_usa = ipc_usa.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el IPC de Estados Unidos ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
ipc_usa["inflacion_usa"] = ipc_usa.iloc[:, 0].pct_change(12) #Calculamos la inflación interanual de Estados Unidos usando iloc para acceder a la primera columna (evita problemas de nombres)
ipc_usa = ipc_usa[["inflacion_usa"]]
ipc_usa = ipc_usa.reset_index()
ipc_usa = ipc_usa.rename(columns={"index": "Date"}) #CORREGIDO: la columna de fecha después de reset_index() se llama "index", no "DATE"

dataset = pd.merge(dataset, ipc_usa, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del IPC de Estados Unidos, uniendo por la columna Date

In [ ]:
dataset.head()

In [ ]:
ipc_eurozona = fred.get_series("CP0000EZ19M086NEST", observation_start=inicio, observation_end=fin)
ipc_eurozona = ipc_eurozona.to_frame()

ipc_eurozona.info()
ipc_eurozona = ipc_eurozona.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el IPC de la eurozona ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
ipc_eurozona["inflation_eu"] = ipc_eurozona.iloc[:, 0].pct_change(12) #Calculamos la inflación interanual de la eurozona usando iloc para acceder a la primera columna 
ipc_eurozona = ipc_eurozona[["inflation_eu"]]
ipc_eurozona = ipc_eurozona.reset_index()
ipc_eurozona = ipc_eurozona.rename(columns={"index": "Date"}) #CORREGIDO: la columna de fecha después de reset_index() se llama "index", no "DATE"

dataset = pd.merge(dataset, ipc_eurozona, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del IPC de la eurozona, uniendo por la columna Date 

In [ ]:
dataset.head()

In [ ]:
desempleoUSA = fred.get_series("UNRATE", observation_start=inicio, observation_end=fin)
desempleoUSA = desempleoUSA.to_frame()

desempleoUSA.info()
desempleoUSA = desempleoUSA.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el desempleo de Estados Unidos ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
desempleoUSA = desempleoUSA.rename(columns={desempleoUSA.columns[0]: "desempleo"})
desempleoUSA = desempleoUSA.reset_index()
desempleoUSA = desempleoUSA.rename(columns={"index": "Date"})

dataset = pd.merge(dataset, desempleoUSA, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del desempleo de Estados Unidos, uniendo por la columna Date

In [ ]:
dataset.tail()

In [ ]:
tipos_ecb = fred.get_series("ECBDFR") #Descargamos los datos de los tipos de interés de la ECB desde la base de datos de FRED
tipos_ecb = tipos_ecb.to_frame() 

tipos_ecb.info()
tipos_ecb = tipos_ecb.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque los tipos de interés de la ECB ya se publican con datos mensuales, pero lo hacemos por consistencia con el resto de variables
tipos_ecb = tipos_ecb.rename(columns={tipos_ecb.columns[0]: "tipos_ecb"})
tipos_ecb = tipos_ecb.reset_index()
tipos_ecb = tipos_ecb.rename(columns={"index":"Date"}) #Renombramos la columna index a Date


In [ ]:
dataset = pd.merge(dataset, tipos_ecb, on="Date", how="inner")

In [ ]:
curva10Y2 = fred.get_series("T10Y2Y") #Descargamos los datos de la curva de tipos de interés a 10 años menos la curva de tipos de interés a 2 años desde la base de datos de FRED
curva10Y2 = curva10Y2.to_frame()

curva10Y2.info()
curva10Y2 = curva10Y2.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el IPC de la eurozona ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
curva10Y2 = curva10Y2.rename(columns={curva10Y2.columns[0]: "curva10Y2Y"})
curva10Y2 = curva10Y2.reset_index()
curva10Y2 = curva10Y2.rename(columns={"index":"Date"})


In [ ]:
dataset = pd.merge(dataset, curva10Y2, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del IPC de la eurozona, uniendo por la columna Date 

In [ ]:
dataset = dataset.rename(columns={"0_x":"tipos_ecb"}) #Renombramos la columna 0_x a tipos_ecb
dataset = dataset.rename(columns={"0_y":"curva10Y2Y"}) #Renombramos la columna 0_y a curva10Y2Y
dataset.tail()
dataset.info()

In [ ]:
activo = 'AAPL'

datos_activo = yf.download(activo, start=inicio, end=fin, interval='1mo')['Close'] #Descargamos los datos de cierre del activo que queramos analizar
datos_activo.info()
datos_activo = datos_activo.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque los datos del activo ya se publican con datos mensuales, pero lo hacemos por consistencia con el resto de variables    
#datos_activo = datos_activo.to_frame()
datos_activo = datos_activo.rename(columns={"AAPL":"precio"})
datos_activo["activo_return"] = datos_activo["precio"].pct_change() #Calculamos el retorno mensual del activo y lo guardamos en una nueva columna

datos_activo['media_movil_3m'] = datos_activo["precio"].rolling(3).mean() #Calculamos la media móvil de 3 meses del precio del activo y la guardamos en una nueva columna
datos_activo['media_movil_6m'] = datos_activo["precio"].rolling(6).mean() #Calculamos la media móvil de 6 meses del precio del activo y la guardamos en una nueva columna
datos_activo['media_movil_12m'] = datos_activo["precio"].rolling(12).mean() #Calculamos la media móvil de 12 meses del precio del activo y la guardamos en una nueva columna

datos_activo['volatilidad_3m'] = datos_activo["activo_return"].rolling(3).std() #Calculamos la volatilidad de 3 meses del precio del activo y la guardamos en una nueva columna
datos_activo['volatilidad_6m'] = datos_activo["activo_return"].rolling(6).std() #Calculamos la volatilidad de 6 meses del precio del activo y la guardamos en una nueva columna

datos_activo['momentum_3m'] = datos_activo["precio"].rolling(3).sum() #Calculamos el momentum de 3 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['momentum_6m'] = datos_activo["precio"].rolling(6).sum() #Calculamos el momentum de 6 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['momentum_12m'] = datos_activo["precio"].rolling(12).sum() #Calculamos el momentum de 12 meses del precio del activo y lo guardamos en una nueva columna

datos_activo['lag_1'] = datos_activo["activo_return"].shift(1) #Calculamos el lag de 1 mes del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_2'] = datos_activo["activo_return"].shift(2) #Calculamos el lag de 2 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_3'] = datos_activo["activo_return"].shift(3) #Calculamos el lag de 3 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_6'] = datos_activo["activo_return"].shift(6) #Calculamos el lag de 6 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_12'] = datos_activo["activo_return"].shift(12) #Calculamos el lag de 12 meses del precio del activo y lo guardamos en una nueva columna
datos_activo.info()

datos_activo = datos_activo.reset_index()

In [ ]:
dataset = pd.merge(dataset, datos_activo, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del activo, uniendo por la columna Date

In [ ]:
dataset = dataset.rename(columns={
    "EURUSD=X": "eurusd",
    "^VIX": "vix"
}) #Renombramos las columnas del eurodolar y del vix para que tengan nombres más cortos y fáciles de manejar
dataset["target"] = dataset["activo_return"].shift(-1) #Creamos la columna target, que es el retorno del activo en el mes siguiente
#dataset.info()
#dataset.iloc[258:]
#dataset.tail()
#dataset.columns

dataset['desempleo'] = imputar_valores_faltantes(dataset, 'desempleo')
dataset['desempleo'] = dataset['desempleo'].round(2)
dataset.isnull().sum()
#dataset.iloc[258:] 

In [ ]:
#dataset['target'].isnull()
#dataset.iloc[:6]
print(dataset.columns)

#Vamos a ver los valores de cada columna, para asi ver posibles anomalias
print(dataset['petroleo_return'].value_counts()) 
print(dataset['oro_return'].value_counts()) 

mask = (dataset['oro_return'] == 0) | (dataset['petroleo_return'] == 0)

#print(dataset[mask].head()) #Mostramos las filas donde el retorno del oro o del petroleo es 0, para ver si hay alguna anomalía en esas fechas
#Una vez arreglado el problema de los 0, vemos que el oro y petroleo return, ya están bien

#print(dataset['petroleo_return'].value_counts()) 
# #dataset['oro_return'].value_counts() 

#dataset[['Date','activo_return', 'target']]


#dataset.dtypes

dataset.sort_values(by="Date", ascending=True, inplace=True) #Ordenamos el dataset por la columna Date para asegurarnos de que los datos están en orden cronológico
#dataset.head()

In [ ]:
#dataset_ml = dataset.copy() #Creamos una copia del dataset para usarlo en el modelo de machine learning, asi mantenemos el dataset original por si queremos hacer alguna otra cosa con él
# Lags de tipos de interés (tienen mucha memoria)
#dataset_ml['tipos_fed_lag1'] = dataset_ml['tipos_fed'].shift(1)
#dataset_ml['tipos_fed_lag3'] = dataset_ml['tipos_fed'].shift(3)
#dataset_ml['tipos_fed_lag6'] = dataset_ml['tipos_fed'].shift(6)

# Lags de inflación (también tiene memoria)
#dataset_ml['inflacion_usa_lag1'] = dataset_ml['inflacion_usa'].shift(1)
#dataset_ml['inflacion_usa_lag3'] = dataset_ml['inflacion_usa'].shift(3)
#dataset_ml['inflacion_usa_lag6'] = dataset_ml['inflacion_usa'].shift(6)

# Lags de VIX (memoria media)
#dataset_ml['vix_lag1'] = dataset_ml['vix'].shift(1)
#dataset_ml['vix_lag2'] = dataset_ml['vix'].shift(2)
#dataset_ml['vix_lag3'] = dataset_ml['vix'].shift(3)

# Lags de desempleo (tiene memoria)
#dataset_ml['desempleo_lag1'] = dataset_ml['desempleo'].shift(1)
#dataset_ml['desempleo_lag3'] = dataset_ml['desempleo'].shift(3)

# Lags de curva de tipos
#dataset_ml['curva10Y2Y_lag1'] = dataset_ml['curva10Y2Y'].shift(1)
#dataset_ml['curva10Y2Y_lag3'] = dataset_ml['curva10Y2Y'].shift(3)

In [ ]:
#print(dataset_ml.info())
#print(dataset_ml.isnull().sum())

In [ ]:
#dataset_ml.head()

In [ ]:
dataset.head()

In [ ]:
dataset.to_csv('dataset_final.csv', index=False)